# NPTEL Study Notes: Introduction to Large Language Models (LLMs)

## Topic: Advanced Prompting Techniques & Model Sensitivity

Welcome to your study notes for this module! In this lecture, we transition from basic prompt components to **advanced prompting techniques** that significantly enhance the reasoning capabilities of LLMs. Furthermore, we shift our focus from evaluating models purely on *accuracy* to evaluating their *robustness and sensitivity* to prompt variations, introducing the **POSIX metric**.

These notes are structured to help you understand the core mechanisms, differentiate between similar algorithms, and tackle NPTEL exam questions confidently.

---

## 1. Advanced Prompting Techniques

Standard prompting (simply providing a question and expecting an answer) often fails for tasks requiring complex reasoning, such as math word problems. Because the model is trained to recognize patterns rather than inherently reason, it may produce an incorrect answer if not guided. To solve this, researchers have developed advanced prompting structures.

### A. Chain of Thought (CoT) Prompting

Instead of providing just input-output pairs (few-shot prompting), CoT provides the model with **intermediate reasoning steps (thoughts)**. By demonstrating how to break down a problem, the model learns to mimic this human-like reasoning process.

* **Mechanism:** The prompt includes an example question, the step-by-step logical breakdown, and the final answer. When given a new question, the model generates its own steps before arriving at the answer.
* **Zero-Shot CoT:** Even without providing explicit examples, you can trigger reasoning by simply appending the magic phrase **"Let's think step by step"** to the prompt. This prompts the model to decompose the problem into sub-steps internally.

### B. Chain of Thought with Self-Consistency

LLMs are described as **"stochastic parrots."**

* *Parrots* because they mimic what they learned during the pre-training step (*Note: corrected from transcription error "pretending step"*).
* *Stochastic* because they are non-deterministic; for a single prompt, they can generate different answers across different instances.

**Self-Consistency** leverages this stochastic nature:

1. Ask the model the exact same question multiple times.
2. Generate multiple, distinct reasoning chains (sets of thoughts).
3. Apply a **voting mechanism** (e.g., majority vote) across the final answers of all generated chains.
4. Select the answer and reasoning chain that received the highest consensus.

### C. Tree of Thoughts (ToT)

ToT treats problem-solving as a search algorithm (specifically, a **branch and bound** approach). It generates an entire tree of reasoning paths from root to leaf, where each path is a hypothesis.

ToT requires three specific modules (which can themselves be language models):

1. **Branching Module:** Decides whether to split a current thought into multiple sub-steps/branches.
2. **Scoring Module:** Evaluates the quality of a specific branch to determine if it is heading in the right direction (positive, negative, or unscored).
3. **Backtracking Module:** Decides if the current path is a dead-end and if the algorithm should backtrack to a previous node to explore a different branch.

> **Worked Example: The Game of 24**
> *Goal:* Use the numbers 4, 9, 10, and 13 with basic arithmetic to get exactly 24. (Rules: A number cannot be reused).
> * **Branch 1:** Subtract 4 from 10 $\rightarrow$ $10 - 4 = 6$.
> * *Remaining numbers:* 6, 9, 13.
> 
> 
> * **Branch 2 (Next Step):** Subtract 9 from 13 $\rightarrow$ $13 - 9 = 4$.
> * *Remaining numbers:* 6, 4.
> 
> 
> * **Branch 3 (Final Step):** Multiply 4 and 6 $\rightarrow$ $4 \times 6 = 24$. (Successful branch).
> * *If a branch yields an unusable intermediate number (e.g., 36), the **Scoring Module** flags it, and the **Backtracking Module** returns to a previous state to try addition or division instead.*
> 
> 

### D. Graph of Thoughts (GoT)

Taking ToT a step further, GoT allows different reasoning chains to be **interconnected**.

* **Mechanism:** Thoughts are not just independent tree branches; they form a graph. You can backtrack, refine, use, reuse, and **combine** existing thoughts.
* It utilizes an **aggregator** to decide whether two independent thoughts can be successfully merged to form a better solution.

---

## 2. Model Robustness: Accuracy vs. Sensitivity

While scaling up model parameters generally improves accuracy, **accuracy alone is not enough**. Models are highly sensitive to the exact phrasing of a prompt.

* **Sensitivity:** How drastically a model's output changes when the input prompt is slightly paraphrased. Higher sensitivity is bad.
* **Robustness:** A robust model gives consistent answers to different paraphrases of the same question.

**The Trade-off:**

* *Model A:* 85% Accuracy, 60% Sensitivity (Highly accurate, but highly unstable).
* *Model B:* 75% Accuracy, 20% Sensitivity (Slightly less accurate, but highly robust).
* *Verdict:* **Model B is preferred.** High sensitivity prevents a model from being deployed reliably, as users cannot predict how a slight change in wording will break the output.

---

## 3. The POSIX Metric: Measuring Prompt Sensitivity

To quantify this sensitivity holistically, researchers developed the **POSIX** (Prompt Sensitivity Index).

### Concept of Intent-Preserving Variations

POSIX evaluates models based on **intent-preserving variations**. This means paraphrasing a prompt without changing its core meaning.

* *Example:* "What is the capital of India?", "Where is the capital of India located?", and "Can you tell me the capital city of India?"
* *Assumption:* For a robust model, the probability of generating a specific response should remain mathematically identical across all intent-preserving variations of the prompt.

### The Four Evaluation Criteria of POSIX

To ensure the POSIX metric was valid, researchers tested it against four criteria. An ideal metric must reflect that sensitivity goes *up* (which is bad) when:

1. **Response Diversity Increases:** The model produces a high number of unique (different) answers for paraphrased prompts.
2. **Response Distribution Entropy Increases:** Entropy measures the unpredictability of a distribution.
* *Low Entropy (Good):* Responses are [2, 2, 2, 2, 3]. One dominant answer.
* *High Entropy (Bad):* Responses are [2, 2, 3, 4, 5]. Responses are scattered and unpredictable. Higher entropy means higher sensitivity.


3. **Semantic Coherence Decreases:** Measured by the cosine similarity between the embeddings of the responses. If cosine similarity is low, the semantic meanings of the answers are completely different, meaning sensitivity is high.
4. **Variance in Confidence Increases:** Measured via the log-likelihood of the generated responses. If a model's confidence fluctuates wildly between paraphrased prompts, it is highly sensitive.

### POSIX Mathematical Formulation

Given a dataset $D$, a model $M$, intent-preserving prompt variations $x_i$ and $x_j$ from a set of variations $X$, and a generated response $y_j$:

The sensitivity focuses on the ratio of probabilities:


$$Ratio = \frac{P(y_j \vert{} x_i)}{P(y_j \vert{} x_j)}$$

If the model is perfectly robust, this ratio should ideally be close to 1 (meaning the probability of generating response $y_j$ is the same regardless of whether prompt $x_i$ or paraphrase $x_j$ was used). In POSIX, this ratio is evaluated across all pairs of variations, normalized by the length of the response $\vert{}y_j\vert{}$, and averaged over the entire dataset $D$. As the POSIX score increases, model sensitivity increases (robustness decreases).

### Key Experimental Findings on Sensitivity

* Increasing model size does **not** guarantee lower sensitivity.
* Instruction fine-tuning (chat models vs. base models) does **not** guarantee lower sensitivity.
* **Few-shot prompting** (providing examples) generally *does* help reduce sensitivity, making models more robust.

*(Note: Next lecture covers Alignment techniques, RLHF (Reinforcement Learning from Human Feedback), and ChatGPT design architecture).*

---

## 4. NPTEL Practice Questions

**Q1. Which advanced prompting technique relies on a "branch and bound" algorithmic setup featuring branching, scoring, and backtracking modules?**
a) Chain of Thought with Self-Consistency
b) Graph of Thoughts
c) Tree of Thoughts
d) Zero-shot Chain of Thought

* **Answer: c) Tree of Thoughts**
* *Explanation:* Tree of Thoughts explores paths like a tree search, explicitly using scoring to evaluate branches and backtracking to return from dead ends.

**Q2. Two models are tested with 5 intent-preserving prompt variations. Model X produces the answers [A, A, A, B, B]. Model Y produces the answers [A, B, C, D, E]. According to response distribution entropy, which model is more sensitive?**
a) Model X
b) Model Y
c) Both are equally sensitive
d) Cannot be determined

* **Answer: b) Model Y**
* *Explanation:* Model Y produces a wider, more uniform distribution of unique answers, leading to higher entropy. Higher entropy correlates with higher prompt sensitivity (lower robustness).

**Q3. In the context of LLMs, why are they sometimes referred to as "stochastic parrots"?**
a) Because they only generate text sequentially using deterministic rules.
b) Because they mimic patterns learned during pre-training and output non-deterministic (randomized) variations.
c) Because they have a tree-based memory structure.
d) Because they require reinforcement learning to speak.

* **Answer: b)**
* *Explanation:* "Parrots" refers to mimicking pre-training data without true reasoning. "Stochastic" refers to the non-deterministic nature where the same prompt can yield different outputs.

**Q4. True or False: According to experimental results with the POSIX metric, larger model sizes and instruction fine-tuning always guarantee lower prompt sensitivity.**

* **Answer: False**
* *Explanation:* The lecture explicitly states there is no clear-cut evidence that larger models or chat/instruct versions automatically reduce sensitivity. However, adding few-shot examples generally does help.

**Q5. When evaluating the Semantic Coherence of responses to test prompt sensitivity, which mathematical metric is primarily used?**
a) Log-likelihood variance
b) Response length normalization
c) Cosine similarity between embeddings
d) Majority voting

* **Answer: c)**
* *Explanation:* Semantic coherence is checked by looking at the cosine similarity between the embeddings of the generated responses. Higher similarity means lower sensitivity.

---

## 5. Quick Revision Summary

* **CoT:** Gives step-by-step reasoning. Zero-shot CoT uses "Let's think step by step".
* **CoT + Self Consistency:** Runs the prompt multiple times and takes a majority vote of the outputs.
* **ToT:** Tree structure using Branching, Scoring, and Backtracking (Branch & Bound).
* **GoT:** Allows thoughts to be interconnected, aggregated, reused, and refined.
* **Sensitivity vs. Accuracy:** A robust model (low sensitivity) is preferred over a highly accurate but unpredictable model.
* **POSIX Metric:** Measures sensitivity using Intent-Preserving Variations.
* **4 POSIX Criteria:** A bad (highly sensitive) model shows **increased** response diversity, **increased** distribution entropy, **decreased** semantic coherence (cosine similarity), and **increased** variance in confidence (log-likelihood).